In [3]:
import pandas as pd
import json
import importlib

# load the data

In [4]:
concept_root = "../data/concept/"
out_concept_root = "../data/outside_concept/"
response_root = "../data/respondent/"

In [5]:
# take the concepts 
with open(concept_root + 'new_cid_concept_us_food.json', 'r') as f:
    food_concepts = json.load(f)

# all concepts 
all_us_food_concepts = pd.read_excel(out_concept_root + '0407_cleaned_us_food_concepts.xlsx')

# open transformed
with open(response_root + 'transformed_0407_id_normal_interview.json', 'r', encoding='utf-8') as f:
    transformed_respondent = json.load(f)

# the similarity function

give a function/ the code, which is the similarity search 

input:
content to search, list of content to be searched, top_n （n most relevant ones）, bottom_m (m least relevant ones)

output: 
top n most relevant 
bottom m least relevant  


the algo should be optimal (I don't mind do the embedding for all the available input first ), be fast. 
use "sentence-transformers/all-MiniLM-L6-v2"


In [6]:
import sys
sys.path.append("../")
from models import similar as sm

c:\Users\Yuding.Duan\OneDrive - Ipsos\3. self_projects\llm_synthetic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
importlib.reload(sm)

<module 'models.similar' from 'c:\\Users\\Yuding.Duan\\OneDrive - Ipsos\\3. self_projects\\llm_synthetic\\combination\\..\\models\\similar.py'>

In [60]:
# Example usage with caching:
corpus = all_us_food_concepts['ConceptText'].tolist()

# Initialize searcher and fit with cache (first run computes, subsequent runs load from cache)
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl"

searcher = sm.SimilaritySearcher()
searcher.fit(corpus, cache_path=CACHE_PATH)  # Embeddings cached to disk

# Search (fast - only query embedding computed)
query = "OIKOS TRIPLE ZERO PLAIN HIGH PROTEIN YOGURT\n\nMore of what you want, less of what you don't - 18g of protein 0% fat, 0g added sugars and 0 artificial sweeteners.\n\nAvailable in 32oz large size, Oikos Triple Zero Plain is a deliciously simple way to get the protein you need. Perfect to add to smoothies, parfaits, or enjoy on it's own!\n\n- 18g Protein\n- 0g Added Sugar\n- 0 Artificial Sweeteners\n- 0% Fat\n- Project Non-GMO Verified\n\n32oz Multi-Serve Tub - $5.99\n\nCurrent Oikos Assortment Still Available"
top_results, bottom_results = searcher.search(query, top_n=10, bottom_m=10)

print("Top 5 most similar:")
for text, score in top_results:
    print(f"  {score:.4f}: {text[:80]}...")

print("\nBottom 5 least similar:")
for text, score in bottom_results:
    print(f"  {score:.4f}: {text[:80]}...")

Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 8037 embeddings loaded from cache
Top 5 most similar:
  0.9960: Oikos Triple Zero Plain High Protein Yogurt More of what you want, less of what ...
  0.9960: Oikos Triple Zero Plain High Protein Yogurt More of what you want, less of what ...
  0.9369: Oikos Anything but Plain High Protein Yogurt More of what you want, less of what...
  0.9369: Oikos Anything but Plain High Protein Yogurt More of what you want, less of what...
  0.8461: OIKOS Triple Zero Mocha Flavored Yogurt STRONGER MAKES EVERYTHING BETTER® OIKOS ...
  0.8185: OIKOS PRO+ FUEL HIGH-PROTEIN YOGURT WITH COMPLEX CARBS TO FUEL YOU Try new Oikos...
  0.8185: OIKOS PRO+ FUEL HIGH-PROTEIN YOGURT WITH COMPLEX CARBS TO FUEL YOU Try new Oikos...
  0.8113: OIKOS PRO BI-LAYER HIGH-PROTEIN YOGURT WITH A DELIGHTFUL CREAM TO REWARD YOUR EF...
  0.8113: OIKOS PRO BI-LAYER HIGH-PROTEIN YOGURT WITH A DELIGHTFUL CREAM TO REWARD YOUR 

# people like or agains 

In [19]:
import pandas as pd
import json
import os

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI  # Requires langchain-openai package
from dotenv import load_dotenv
load_dotenv()
lite_llm_key_all = os.getenv('LITE_LLM_KEY_ALL')

llm_model = ChatOpenAI(base_url='https://ipsos.litellm-prod.ai/',model='gpt-5', temperature=0.1, api_key=lite_llm_key_all)

1. giving a concept
2. it's needs, chain of thought, would u think he will like it ?  
3. it shall be able to process a list 
that's it 

In [8]:
from models import need_filter as nf

In [14]:
import importlib
importlib.reload(nf)

<module 'models.need_filter' from 'c:\\Users\\Yuding.Duan\\OneDrive - Ipsos\\3. self_projects\\llm_synthetic\\combination\\..\\models\\need_filter.py'>

In [15]:
import random
# random individual 
random.seed(42)  # For reproducibility
n = 10
ids = list(transformed_respondent.keys()); selected_ids = random.sample(ids, n)
# kpi 
kpis = random.choices(["relevance", "differentiation", "believability"], k=n)

# concepts 
concepts = random.choices(all_us_food_concepts['ConceptText'].tolist(), k=n)

In [ ]:
# Prepare batch items

##### this is only for kinda dataframe generation. 

reasoning = False
items = []
qneeds_texts = []

for resp_id, kpi, concept in zip(selected_ids, kpis, concepts):
    respondent_info = transformed_respondent[resp_id]

    # Get qneeds for display
    qneeds = []
    for q in ['qneed2', 'qneed3']:
        if q in respondent_info:
            qneeds.append(f"{respondent_info[q]['cate']}: {respondent_info[q]['comment']}")
    qneeds_texts.append("\n".join(qneeds))

    items.append({
        "new_concept": concept,
        "kpi_type": kpi,
        "system_info": respondent_info,
        "return_reasoning": reasoning
    })

# Run batch processing concurrently (use await in notebooks)
print(f"Processing {len(items)} items concurrently...")
batch_results = await nf.ai_filter_batch_async(
    items,
    max_concurrency=4,
    show_progress=True,
    progress_desc="AI filter"
)
print("Done!")

# Collect results
results = {
    'respondent_id': selected_ids,
    'respondent_needs': qneeds_texts,
    'kpi_type': kpis,
    'concept': [c[:500] for c in concepts],  # truncate for readability
    'predicted_answer': [r['answer'] if reasoning else r for r in batch_results],
    'reasoning': [r['reasoning'] if reasoning else '' for r in batch_results]
}

# Create DataFrame and export to Excel
results_df = pd.DataFrame(results)
output_path = "../data/sample_ai_filter_results_reasoning.xlsx"
results_df.to_excel(output_path, index=False)
print(f"Results saved to {output_path}")

Processing 10 items concurrently...


AI filter: 100%|██████████| 10/10 [00:18<00:00,  1.88s/it]

Done!
Results saved to ../data/sample_ai_filter_results_reasoning.xlsx


# go for the whole process

**prepare all the data**

- food_concepts  
- all_us_food_concepts  
- transformed_respondent

In [ ]:
us_food_cates = list(set(all_us_food_concepts.dropna(subset=['CAT2'])['CAT2']))


# Example usage with caching:
corpus = us_food_cates[:]

# Initialize searcher and fit with cache (first run computes, subsequent runs load from cache)
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl"



# top_results, bottom_results = searcher_cate.search(query, top_n=5, bottom_m=0)

In [ ]:
all_us_food_concepts.drop_duplicates(subset=['ConceptText'], inplace=True)
all_us_food_concepts.reset_index(drop=True, inplace=True)

In [18]:
response_table = pd.read_excel(response_root + "response_table_us_food.xlsx")

In [11]:
kpi_mapping = {
    "relevance": "RelFlag",
    "differentiation": "DiffFlag",
    "believability": "BelFlag"
}

let's go

In [12]:
response_table.drop(columns=['id'], inplace=True)

**ob_type**: `real` or `synthetic_contra` or `synthetic_same`

In [ ]:
# parameters 
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl" # vector cache path
cate_match_bound = 0.5 # this is for category matching
searcher_cate = sm.SimilaritySearcher()
searcher_cate.fit(us_food_cates, cache_path=CACHE_PATH)


## boundary stuff 
contra_lower_bound = 0.28

same_lower_bound = 0.5
smae_upper_bound = 0.75

Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 22 embeddings loaded from cache


In [121]:
final_result = {'ids':[], 'concept':[], 'question':[], 'answer':[], 'ob_type':[], 'corresponding_concept':[]}
final_results = [] 

In [145]:
response_table.loc[(response_table['question']==kpi) & (response_table['answer']=='no')]

,id,ids,concept,question,answer
44,26,2cbd4c90-2b45-11f0-a08f-8b651dedd23a,2,differentiation,no
45,26,2cbd4c90-2b45-11f0-a08f-8b651dedd23a,4,differentiation,no
63,32,b01f6cb0-0754-11f0-a000-d10a7e1714e4,9,differentiation,no
74,35,3f833900-d75f-11f0-9a9d-29faafdd74dd,4,differentiation,no
75,35,3f833900-d75f-11f0-9a9d-29faafdd74dd,5,differentiation,no
...,...,...,...,...,...
4689,2355,7f991c60-b828-11ef-9340-bba6897203d9,6,differentiation,no
4700,2364,ab9fd2c0-cc8e-11ef-80bd-0b93f20d4d9f,1,differentiation,no
4701,2364,ab9fd2c0-cc8e-11ef-80bd-0b93f20d4d9f,9,differentiation,no
4705,2371,c04fc000-1403-11f0-be11-751b713de3e5,4,differentiation,no


In [150]:
# minimum unit 
id = "b01f6cb0-0754-11f0-a000-d10a7e1714e4"
kpi = "differentiation"



mask = (response_table['question'] == kpi) &(response_table['ids']==id)
sub = response_table[mask]
sub['ob_type'] = 'real'
sub['corresponding_concept'] = sub['concept']
final_results.append(sub)

C:\Users\Yuding.Duan\AppData\Local\Temp\ipykernel_32184\3734395815.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['ob_type'] = 'real'
C:\Users\Yuding.Duan\AppData\Local\Temp\ipykernel_32184\3734395815.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['corresponding_concept'] = sub['concept']


In [151]:
sub

,id,ids,concept,question,answer,ob_type,corresponding_concept
62,32,b01f6cb0-0754-11f0-a000-d10a7e1714e4,3,differentiation,yes,real,3
63,32,b01f6cb0-0754-11f0-a000-d10a7e1714e4,9,differentiation,no,real,9


In [152]:
if len(sub['answer'].value_counts()) == 1:
    process_type = 'contra' # to get the opposite answer
else:
    process_type = 'same' # to get the same answer

In [153]:
food_concepts[str(item['concept'])]['concept_Cate']

'Bakery items (shelf stable) '

In [154]:
top_results

[('Indulge in the Irresistible: Butter Meets Cinnamon in Every Bite! You love the taste of creamy butter on deliciously sweet cinnamon bread. Imagine that butter already baked in, giving you this amazing combination in every slice right out of the bag. Introducing Thomas’ Butter &#38  Cinnamon Swirl! With Thomas’ Butter &#38  Cinnamon Swirl you can indulge in the delightful pairing of rich butter and aromatic cinnamon. This new Swirl bread now comes in a thicker slice for more flavor in every bite. Start your day with Thomas’ Butter &#38  Cinnamon Swirl, a convenient and delicious breakfast option you can savor to start your day right. $4.79',
  np.float32(0.98954797)),
 ('Introducing Thomas’ Croissant Toast: A new way to enjoy the Croissant Experience Enjoy the buttery, tender, flakiness of fresh croissants in delicious golden slices. New Thomas’ Croissant Toast gives you more opportunities to enjoy the deliciousness of croissants every day . Croissant Toast is sliced, so it’s easy to

In [155]:

for i in range(len(sub)):
    item = sub.iloc[i]

    # to get the category 
    if process_type == 'contra':
        query = food_concepts[str(item['concept'])]['concept_Cate']
        top_results, bottom_results = searcher_cate.search(query, top_n=5, bottom_m=0)
        suitable_cates = set([k[0] for k in top_results if k[1] >= cate_match_bound])

        if item['answer'] == 'yes':
            out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['L', 'ML']) & (all_us_food_concepts['CAT2'].isin(suitable_cates))
        else:
            out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['H', 'MH']) & (all_us_food_concepts['CAT2'].isin(suitable_cates))
        
        filted_concepts = all_us_food_concepts[out_mask]
        corpus = list(filted_concepts['ConceptText'])


        searcher_concept = sm.SimilaritySearcher()
        searcher_concept.fit(corpus, cache_path=CACHE_PATH)

        query = food_concepts[str(item['concept'])]['concept_content']
        top_results, bottom_results = searcher_concept.search(query, top_n=5, bottom_m=5)
        candidate = [item[0] for item  in bottom_results if item[1]<= contra_lower_bound]
        if not candidate:
            continue

        items = []
        respondent_info = transformed_respondent[id]
        reasoning = False

        for concept in candidate:
            items.append({
                "new_concept": concept,
                "kpi_type": kpi,
                "system_info": respondent_info,
                "return_reasoning": reasoning
            })

        # Run batch processing concurrently (use await in notebooks)
        print(f"Processing {len(items)} items concurrently...")
        batch_results = await nf.ai_filter_batch_async(
            items,
            max_concurrency=4,
            show_progress=True,
            progress_desc="AI filter"
        )
        
        if item['answer'] == "yes":
            pick_answer = 'no'    
        else:
            pick_answer = 'yes'
        
        for k in range(len(batch_results)):
            if batch_results[k] == pick_answer:
                final_result['ids'].append(id)
                final_result['concept'].append(candidate[k])
                final_result['question'].append(kpi)
                final_result['answer'].append(batch_results[k])
                final_result['ob_type'].append('synthetic_contra')
                final_result['corresponding_concept'].append(str(item['concept']))
    else: 

        if item['answer'] == 'yes':
            out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['H', 'MH'])
        else:
            out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['L', 'ML']) 
        
        filted_concepts = all_us_food_concepts[out_mask]
        corpus = list(filted_concepts['ConceptText'])


        searcher_concept = sm.SimilaritySearcher()
        searcher_concept.fit(corpus, cache_path=CACHE_PATH)

        query = food_concepts[str(item['concept'])]['concept_content']
        top_results, bottom_results = searcher_concept.search(query, top_n=8, bottom_m=5)
        candidate = [item[0] for item  in top_results if same_lower_bound<= item[1]<= smae_upper_bound]
        if not candidate:
            continue

        items = []
        respondent_info = transformed_respondent[id]
        reasoning = False

        for concept in candidate:
            items.append({
                "new_concept": concept,
                "kpi_type": kpi,
                "system_info": respondent_info,
                "return_reasoning": reasoning
            })

        # Run batch processing concurrently (use await in notebooks)
        print(f"Processing {len(items)} items concurrently...")
        batch_results = await nf.ai_filter_batch_async(
            items,
            max_concurrency=4,
            show_progress=True,
            progress_desc="AI filter"
        )
        
        pick_answer = item['answer']
        for k in range(len(batch_results)):
            if batch_results[k] == pick_answer:
                final_result['ids'].append(id)
                final_result['concept'].append(candidate[k])
                final_result['question'].append(kpi)
                final_result['answer'].append(batch_results[k])
                final_result['ob_type'].append('synthetic_same')
                final_result['corresponding_concept'].append(str(item['concept']))


Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 1993 embeddings loaded from cache
Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 531 embeddings loaded from cache
Processing 5 items concurrently...


AI filter: 100%|██████████| 5/5 [00:26<00:00,  5.27s/it]


In [156]:
fr_df = pd.DataFrame(final_result)